# Flight Delay Analysis - API Data Pull

**Goal:** Pull live flight and airline data from AviationStack and OpenSky

**APIs Used:** AviationStack (flight status), OpenSky Network (aircraft tracking)

**Primary Data:** BTS Official airline on-time performance data (7M+ rows)

In [1]:
import os
import requests
import pandas as pd
import json
from dotenv import load_dotenv

load_dotenv()

AVIATIONSTACK_KEY = os.getenv('AVIATIONSTACK_KEY')

print('Environment loaded successfully')

def get_airlines():
    url = 'http://api.aviationstack.com/v1/airlines'
    params = {'access_key': AVIATIONSTACK_KEY, 'limit': 100}
    r = requests.get(url, params=params)
    return r.json()['data']
 
airlines_raw = get_airlines()
df_airlines = pd.json_normalize(airlines_raw)
df_airlines.to_csv('/Users/NilkanthPatel2/Desktop/Future/Projects/flight-delay-analysis/data/raw/airlines_reference.csv', index=False)
print(f'Saved {len(df_airlines)} airlines')
df_airlines[['airline_name','iata_code','country_name']].head(10)



def get_airports(country_code='US'):
    url = 'http://api.aviationstack.com/v1/airports'
    params = {
        'access_key': AVIATIONSTACK_KEY,
        'country_iso2': country_code,
        'limit': 100
    }
    r = requests.get(url, params=params)
    return r.json().get('data', [])
 
airports_raw = get_airports()
df_airports = pd.json_normalize(airports_raw)
df_airports.to_csv('/Users/NilkanthPatel2/Desktop/Future/Projects/flight-delay-analysis/data/raw/airports_reference.csv', index=False)
print(f'Saved {len(df_airports)} US airports')
df_airports[['airport_name','iata_code','city_iata_code','country_name']].head(10)



def get_delayed_flights(min_delay=30):
    url = 'http://api.aviationstack.com/v1/flights'
    params = {
        'access_key': AVIATIONSTACK_KEY,
        'dep_iata': 'ORD',   # Chicago O'Hare — one of the busiest delay hubs
        'flight_status': 'active',
        'limit': 100
    }
    r = requests.get(url, params=params)
    data = r.json().get('data', [])
    return pd.json_normalize(data)
 
df_live = get_delayed_flights()
df_live.to_csv('/Users/NilkanthPatel2/Desktop/Future/Projects/flight-delay-analysis/data/raw/live_flights_sample.csv', index=False)
print(f'Pulled {len(df_live)} live flights')
print(df_live.columns.tolist())


def get_opensky_arrivals(airport_icao='KORD', hours_back=2):
    """Pull recent arrivals at a major US airport."""
    import time
    end_time   = int(time.time())
    begin_time = end_time - (hours_back * 3600)
 
    url = 'https://opensky-network.org/api/flights/arrival'
    params = {
        'airport': airport_icao,   # KORD = Chicago O'Hare ICAO code
        'begin': begin_time,
        'end': end_time
    }
    r = requests.get(url, params=params)
    if r.status_code == 200:
        return pd.DataFrame(r.json())
    else:
        print('Status:', r.status_code, r.text)
        return pd.DataFrame()
 
# Pull arrivals for the three most delay-prone airports
airports_to_pull = {
    'KORD': 'Chicago OHare',
    'KATL': 'Atlanta Hartsfield',
    'KLGA': 'New York LaGuardia'
}


all_arrivals = []
for icao, name in airports_to_pull.items():
    df_arr = get_opensky_arrivals(icao)
    df_arr['airport_name'] = name
    all_arrivals.append(df_arr)
    print(f'Pulled {len(df_arr)} arrivals for {name}')
 
df_opensky = pd.concat(all_arrivals, ignore_index=True)
df_opensky.to_csv('/Users/NilkanthPatel2/Desktop/Future/Projects/flight-delay-analysis/data/raw/opensky_arrivals.csv', index=False)
print(f'Total arrival records: {len(df_opensky)}')



Environment loaded successfully
Saved 100 airlines
Saved 100 US airports
Pulled 77 live flights
['flight_date', 'flight_status', 'live', 'departure.airport', 'departure.timezone', 'departure.iata', 'departure.icao', 'departure.terminal', 'departure.gate', 'departure.delay', 'departure.scheduled', 'departure.estimated', 'departure.actual', 'departure.estimated_runway', 'departure.actual_runway', 'arrival.airport', 'arrival.timezone', 'arrival.iata', 'arrival.icao', 'arrival.terminal', 'arrival.gate', 'arrival.baggage', 'arrival.scheduled', 'arrival.delay', 'arrival.estimated', 'arrival.actual', 'arrival.estimated_runway', 'arrival.actual_runway', 'airline.name', 'airline.iata', 'airline.icao', 'flight.number', 'flight.iata', 'flight.icao', 'flight.codeshared.airline_name', 'flight.codeshared.airline_iata', 'flight.codeshared.airline_icao', 'flight.codeshared.flight_number', 'flight.codeshared.flight_iata', 'flight.codeshared.flight_icao', 'aircraft.registration', 'aircraft.iata', 'aircr